In [50]:
from datetime import datetime
from pathlib import Path
import joblib
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import ParameterGrid
import src.fda.kde.estimators as kde
import matplotlib.pyplot as plt

import sys
sys.path.append("..")
import src.fda.utils             as fdaUtils
import src.forecasting.accuracy  as acc 

In [62]:
EXECUTION_DATE : str = datetime.now().strftime('%Y%m%d')
# EXECUTION_DATE = "20260611"
EXECUTION_DATE = "20260612"

CV_MAIN_PATH   :  str = f'../data/interim/cv/{EXECUTION_DATE}/'
CV_RESULTS_PATH : str = f'../data/interim/cv/{EXECUTION_DATE}/results/'
Path(CV_MAIN_PATH).mkdir(parents=True, exist_ok=True)
Path(CV_RESULTS_PATH).mkdir(parents=True, exist_ok=True)

# Parameters

In [63]:
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df_returns = pd.read_excel(returns_path, index_col="time")
df_returns

,2024-12-02,2024-12-03,2024-12-04,2024-12-05,2024-12-06,2024-12-09,2024-12-10,2024-12-11,2024-12-12,2024-12-13,...,2025-11-14,2025-11-17,2025-11-18,2025-11-19,2025-11-21,2025-11-24,2025-11-25,2025-11-26,2025-11-27,2025-11-28
time,,,,,,,,,,,,,,,,,,,,,
10:05:00,-0.002271,0.003372,0.001716,0.005538,0.000069,0.006429,0.002343,0.001918,-0.007582,-0.000413,...,-0.001522,0.000739,-0.000613,0.000306,-0.002021,-0.000810,0.001557,-0.001565,-0.000193,0.002906
10:10:00,-0.000582,0.002832,-0.000502,0.000739,0.000261,0.001891,0.000775,-0.001172,0.000216,-0.000432,...,0.000360,-0.000105,-0.000159,-0.000362,0.000092,0.001131,0.001170,0.000202,0.000556,0.000475
10:15:00,-0.001997,0.001091,0.000111,-0.000192,0.000407,-0.001392,-0.000477,-0.001526,0.000198,-0.001821,...,0.001121,-0.000284,0.003272,-0.001107,-0.001012,-0.000174,0.001002,0.001815,-0.000168,0.000973
10:20:00,-0.000037,-0.000505,-0.000327,-0.000580,-0.000436,-0.000405,0.001263,0.000676,-0.002667,0.001033,...,0.000449,0.000335,0.000853,0.000981,-0.000057,-0.000096,0.000832,0.001951,0.001060,-0.001004
10:25:00,0.000364,-0.001532,0.000429,-0.000426,-0.000162,-0.000130,0.000060,0.000818,-0.000970,0.000670,...,-0.000871,-0.000559,0.001030,-0.000749,-0.001194,0.001338,0.000128,0.001279,-0.000162,-0.000651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16:40:00,-0.000169,0.000195,-0.000833,0.000137,0.000127,0.000062,0.000253,-0.003736,0.000405,0.000168,...,-0.000255,-0.000412,-0.000398,-0.000301,-0.000428,-0.000232,0.000473,-0.000521,0.000118,-0.000099
16:45:00,-0.000246,-0.000226,-0.000897,0.000536,0.001010,0.000047,-0.000063,0.002746,-0.000154,-0.000783,...,0.000344,0.000610,0.000224,-0.000353,0.000034,0.000054,-0.000519,0.000386,-0.000085,-0.000468
16:50:00,-0.000512,-0.000546,0.000553,0.000278,-0.000393,-0.000625,0.000141,-0.002952,0.002424,0.000244,...,0.000202,-0.000350,-0.000240,0.000239,-0.000314,-0.000175,0.000022,0.000298,0.000399,-0.000041


## KDE

In [64]:
t_dfs = range(3,6)

# 1. Define the parameter universes for each method
rot_grid = {
    "method": ["rot"],
    "kernel": ["gaussian", "epanechnikov"], #"tophat", "exponential", "linear", "cosine"],
    "sigma_robust": [True, False]
}

lcv_sklearn_grid = {
    "method": ["cross_validate"],
    "kernel": ["gaussian"], #"epanechnikov"], #"tophat", "exponential", "linear", "cosine"],
    "cv": ["LOO"]#[5]
}

adaptive_grid = {
    "method": ["adaptive"], #"dpi"],
    "kernel": ["gaussian", "epanechnikov"],
    # "h": kernel_rules
}

lcv_t_grid = {
    "method": ["cross_validate"],
    "kernel": ["t-student"],
    "df": [df for df in t_dfs],
    "cv": ["LOO"]#, "LOO"] [5]
}

kernel_rules = np.unique([v for d in kde.get_rot_bandwidth_ck().values() for v in d.keys()])

adaptive_grid_t = {
    "method": ["adaptive"], #"dpi"],
    "kernel": ["t-student"],
    "df": [df for df in t_dfs],
    # "h": kernel_rules
}

# 2. Combine grids and build the dictionary
density_param_grid = {}
for grid in [rot_grid, lcv_sklearn_grid, lcv_t_grid, adaptive_grid, adaptive_grid_t]:
    for params in ParameterGrid(grid):
        # Create a unique key based on the parameters
        # e.g., "rot_gaussian_robust_True" or "lcv_t-student_df_3"
        key_parts = [params["kernel"].replace("_","")]
        if "method" in params: key_parts.append(params["method"])
        if "df" in params: key_parts.append(f"df={params['df']}")
        if params.get("sigma_robust"): key_parts.append("robust")
        if params.get("cv") == "LOO": key_parts.append("loo")
        
        # ADD THESE TWO LINES:
        if "alpha" in params: key_parts.append(f"a{params['alpha']}")
        if "h" in params: key_parts.append(f"h{params['h']}") # Distinguish different pilot h values
        
        name = "_".join(key_parts).replace(".", "") # Remove dots for cleaner keys
        density_param_grid[name] = params

print("KDE models:")
for name, params in density_param_grid.items():
    print("\t",name, ":", params)

KDE models:
	 gaussian_rot_robust : {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': True}
	 gaussian_rot : {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': False}
	 epanechnikov_rot_robust : {'kernel': 'epanechnikov', 'method': 'rot', 'sigma_robust': True}
	 epanechnikov_rot : {'kernel': 'epanechnikov', 'method': 'rot', 'sigma_robust': False}
	 gaussian_cross_validate_loo : {'cv': 'LOO', 'kernel': 'gaussian', 'method': 'cross_validate'}
	 t-student_cross_validate_df=3_loo : {'cv': 'LOO', 'df': 3, 'kernel': 't-student', 'method': 'cross_validate'}
	 t-student_cross_validate_df=4_loo : {'cv': 'LOO', 'df': 4, 'kernel': 't-student', 'method': 'cross_validate'}
	 t-student_cross_validate_df=5_loo : {'cv': 'LOO', 'df': 5, 'kernel': 't-student', 'method': 'cross_validate'}
	 gaussian_adaptive : {'kernel': 'gaussian', 'method': 'adaptive'}
	 epanechnikov_adaptive : {'kernel': 'epanechnikov', 'method': 'adaptive'}
	 t-student_adaptive_df=3 : {'df': 3, 'kernel': 't-student', 'meth

## Postprocessing

In [65]:
# Density normalization
# 1. Define your sub-grids to avoid invalid combinations
weigh_grids = [
    {
        'mean_mode': [False, 'expanding'], #'full'],
        'window': [None]
    },
    {
        'mean_mode': ['rolling'],
        'window': [5, 21, 63] # week, month, quarter
    }
]

# 2. Build the dictionary with unique keys
weigh_param_grid = {}
for grid in weigh_grids:
    for params in ParameterGrid(grid):
        # Start the key with the mode
        key_parts = [str(params["mean_mode"])]
        
        # Add the window to the key if it exists (for rolling)
        if params["window"] is not None:
            key_parts.append(f"w{params['window']}")
        
        # Create a clean string key: e.g., "rolling_w5" or "expanding"
        name = "_".join(key_parts)
        weigh_param_grid[name] = params

print("Posprocessing models:")
for name, params in weigh_param_grid.items():
    print("\t",name, ":", params)

Posprocessing models:
	 False : {'mean_mode': False, 'window': None}
	 expanding : {'mean_mode': 'expanding', 'window': None}
	 rolling_w5 : {'mean_mode': 'rolling', 'window': 5}
	 rolling_w21 : {'mean_mode': 'rolling', 'window': 21}
	 rolling_w63 : {'mean_mode': 'rolling', 'window': 63}


In [66]:
theta = []

for kde_name, kde_params in density_param_grid.items():
    for post_name, post_params in weigh_param_grid.items():
        theta.append({
            "kde_name": kde_name,
            **kde_params,
            "postprocess_name": post_name,
            **post_params
        })

print("Total KDE configurations:", len(theta))

Total KDE configurations: 65


In [24]:
kde_databases = {}

for name, params in density_param_grid.items():
    print(f"Processing configuration: {name}")
    
    # bandwidths
    df_h = kde.df_bandwidth_selector(df_returns, **params)    
    kde_params = {k: v for k, v in params.items() if k in ['kernel', 'df']}
    
    # kdes
    df_grids, df_densities = kde.df_to_kde(
        X=df_returns, 
        h=df_h, 
        normalize_densities=False,
        **kde_params
    )

    # postprocessing
    for pp_name, pp_params in weigh_param_grid.items():
        database_name = "___".join([name, pp_name])
        print(f"\t\t{database_name}")
        if pp_name == 'False':
            pass
        else:
            base_grid = np.linspace(df_grids.min().min(), df_grids.max().max(), 5001)
            df_densities = kde.weigh_norm_densities(
                                            df_densities=df_densities,
                                            support=base_grid,
                                            **pp_params
            )


        kde_databases[database_name]= {
            "df_h":         df_h,
            "df_grids":     df_grids,
            "df_densities": df_densities
        }

Processing configuration: gaussian_rot_robust
		gaussian_rot_robust___False
		gaussian_rot_robust___expanding
		gaussian_rot_robust___rolling_w5
		gaussian_rot_robust___rolling_w21
		gaussian_rot_robust___rolling_w63
Processing configuration: gaussian_rot
		gaussian_rot___False
		gaussian_rot___expanding
		gaussian_rot___rolling_w5
		gaussian_rot___rolling_w21
		gaussian_rot___rolling_w63
Processing configuration: epanechnikov_rot_robust
		epanechnikov_rot_robust___False
		epanechnikov_rot_robust___expanding
		epanechnikov_rot_robust___rolling_w5
		epanechnikov_rot_robust___rolling_w21
		epanechnikov_rot_robust___rolling_w63
Processing configuration: epanechnikov_rot
		epanechnikov_rot___False
		epanechnikov_rot___expanding
		epanechnikov_rot___rolling_w5
		epanechnikov_rot___rolling_w21
		epanechnikov_rot___rolling_w63
Processing configuration: gaussian_cross_validate_loo
		gaussian_cross_validate_loo___False
		gaussian_cross_validate_loo___expanding
		gaussian_cross_validate_loo___ro

In [67]:
# Combines CV
cv_path = CV_RESULTS_PATH

all_cvs = []
for model_cv_file in os.listdir(cv_path):
    filepath = '/'.join([cv_path, model_cv_file])
    model_cv = joblib.load(filepath)
    model_cv["fc_date"] = model_cv.apply(lambda x: x.df_support.iloc[:,0].name, axis=1)
    # model_cv["kde_pp_params"] = model_cv["kde_params"].iloc[0] + "___" + model_cv["portprocessing_method"].iloc[0]
    all_cvs.append(model_cv)


df_all_cvs = pd.concat(all_cvs, axis=0)
df_all_cvs = df_all_cvs[["name", "method", "kde_params", "postprocessing_method", "dFPC_dimensions", "etahat_fc_model", "etahat_fc_model_spec", "etahat_fc_model_lags", "fold", "fc_date", "df_support", "df_forecast"]]
# joblib.dump(df_all_cvs, f'{CV_MAIN_PATH}cv_results.jbl')

# df_all_cvs = joblib.load(f'{CV_MAIN_PATH}cv_results.jbl')
df_all_cvs.head()

,name,method,kde_params,postprocessing_method,dFPC_dimensions,etahat_fc_model,etahat_fc_model_spec,etahat_fc_model_lags,fold,fc_date,df_support,df_forecast
2,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,2,1,2025-05-05,2025-05-05 0 -0.018871 1 -0.01...,2025-05-05 0 0.0 1 ...
5,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,2,2,2025-05-06,2025-05-06 0 -0.018871 1 -0.01...,2025-05-06 0 0.0 1 ...
8,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,2,3,2025-05-07,2025-05-07 0 -0.018871 1 -0.01...,2025-05-07 0 0.0 1 ...
11,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,2,4,2025-05-08,2025-05-08 0 -0.018871 1 -0.01...,2025-05-08 0 0.0 1 ...
14,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,2,5,2025-05-09,2025-05-09 0 -0.018871 1 -0.01...,2025-05-09 0 0.0 1 ...


In [68]:
df_all_cvs[df_all_cvs.name=="gaussian_rot_robust___False___2___regression_adalasso___1"]

,name,method,kde_params,postprocessing_method,dFPC_dimensions,etahat_fc_model,etahat_fc_model_spec,etahat_fc_model_lags,fold,fc_date,df_support,df_forecast
2,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,1,2025-05-05,2025-05-05 0 -0.018871 1 -0.01...,2025-05-05 0 0.0 1 ...
5,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,2,2025-05-06,2025-05-06 0 -0.018871 1 -0.01...,2025-05-06 0 0.0 1 ...
8,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,3,2025-05-07,2025-05-07 0 -0.018871 1 -0.01...,2025-05-07 0 0.0 1 ...
11,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,4,2025-05-08,2025-05-08 0 -0.018871 1 -0.01...,2025-05-08 0 0.0 1 ...
14,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,5,2025-05-09,2025-05-09 0 -0.018871 1 -0.01...,2025-05-09 0 0.0 1 ...
...,...,...,...,...,...,...,...,...,...,...,...,...
230,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,77,2025-08-21,2025-08-21 0 -0.018871 1 -0.01...,2025-08-21 0 0.0 1 ...
233,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,78,2025-08-22,2025-08-22 0 -0.018871 1 -0.01...,2025-08-22 0 0.0 1 ...
236,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,79,2025-08-25,2025-08-25 0 -0.018871 1 -0.01...,2025-08-25 0 0.0 1 ...
239,gaussian_rot_robust___False___2___regression_a...,KLE,gaussian_rot_robust,False,2,regression,adalasso,1,80,2025-08-26,2025-08-26 0 -0.018871 1 -0.01...,2025-08-26 0 0.0 1 ...


In [69]:
# Builds loss data for theta_j (kde database) VS theta_i (kde forecast)
kde_base_models = kde_databases.keys()
kde_fc_models = df_all_cvs.name.unique()
records = []
for kde_base_model in kde_base_models:
    kde_model_database = kde_databases[kde_base_model]
    kde_support = kde_databases[kde_base_model]["df_grids"]
    kde_density = kde_databases[kde_base_model]["df_densities"]

    for kde_fc_model in kde_fc_models:
        subset = df_all_cvs[df_all_cvs.name == kde_fc_model]

        for i in range(len(subset)):
            fc_date:    str       = subset['fc_date'].iloc[i]
            fc_support_t: pd.Series = subset['df_support'].iloc[i].iloc[:,0]    
            fc_density_t: pd.Series = subset['df_forecast'].iloc[i].iloc[:,0]

            kde_support_t = kde_support.loc[:,fc_date]
            kde_density_t = kde_density.loc[:,fc_date]

            support_t_interpolated, kde_density_t_interpolated, fc_density_t_interpolated = fdaUtils.align_and_normalize_density(
                                                    kde_support_t, 
                                                    kde_density_t, 
                                                    fc_support_t, 
                                                    fc_density_t
                                                    )
            
            # residuals = kde_density_t_interpolated - fc_density_t_interpolated 
            
            metrics = acc.get_metrics(kde_density_t_interpolated, fc_density_t_interpolated)

            records.append({
                "kde_model": kde_base_model,
                "forecast_model": kde_fc_model,
                "forecast_kde_params": subset["kde_params"].iloc[0],
                "forecast_pp_params": subset["postprocessing_method"].iloc[0],
                "forecast_dFPC_dimension": subset["dFPC_dimensions"].iloc[0],
                "etahat_fc_model": subset["etahat_fc_model"].iloc[0], 
                "etahat_fc_model_spec": subset["etahat_fc_model_spec"].iloc[0], 
                "etahat_fc_model_lags": subset["etahat_fc_model_lags"].iloc[0],
                "date": fc_date,
                # "residuals": residuals,
                **metrics
            })

            # Comparing forecast with KDE
            # plt.figure(figsize=(15,3))
            # plt.plot(support_t_interpolated, kde_density_t_interpolated, label="kde")
            # plt.plot(support_t_interpolated, fc_density_t_interpolated, label="forecast")
            # KLD_t = df_results["KLD"]
            # plt.title(f"{kde_base_model} vs {kde_fc_model} (sKLD={KLD_t})")
            # plt.legend()
            # plt.show()

df_loss_base_long = pd.DataFrame(records)

# joblib.dump(df_loss_base_long, f'{CV_MAIN_PATH}cv_results_cross_comparison_preview.jbl')

In [70]:
df_loss_base_long.head()

,kde_model,forecast_model,forecast_kde_params,forecast_pp_params,forecast_dFPC_dimension,etahat_fc_model,etahat_fc_model_spec,etahat_fc_model_lags,date,KLD,JSD,L1_norm,L2_norm,LINF_norm
0,gaussian_rot_robust___False,gaussian_rot_robust___False___2___regression_a...,gaussian_rot_robust,False,2,regression,adalasso,2,2025-05-05,0.256265,0.022158,28020.473843,1322.594975,111.765819
1,gaussian_rot_robust___False,gaussian_rot_robust___False___2___regression_a...,gaussian_rot_robust,False,2,regression,adalasso,2,2025-05-06,0.036772,0.005600,14055.919803,760.697469,76.068769
2,gaussian_rot_robust___False,gaussian_rot_robust___False___2___regression_a...,gaussian_rot_robust,False,2,regression,adalasso,2,2025-05-07,0.179070,0.005726,14094.743258,662.283626,47.427211
3,gaussian_rot_robust___False,gaussian_rot_robust___False___2___regression_a...,gaussian_rot_robust,False,2,regression,adalasso,2,2025-05-08,0.412885,0.010688,27810.902938,1250.647938,118.638190
4,gaussian_rot_robust___False,gaussian_rot_robust___False___2___regression_a...,gaussian_rot_robust,False,2,regression,adalasso,2,2025-05-09,0.224174,0.017197,33265.165726,2207.725042,328.353996


In [71]:
metrics = ["KLD", "JSD", "L1_norm", "L2_norm", "LINF_norm"]

In [72]:
df_loss_base_long.groupby("etahat_fc_model")[metrics].mean()

,KLD,JSD,L1_norm,L2_norm,LINF_norm
etahat_fc_model,,,,,
VAR,2.272059,0.165739,91423.948742,6240.228457,971.249821
regression,2.353616,0.170927,93419.303502,6347.869095,985.606512


In [73]:
df_loss_base_long.groupby("etahat_fc_model_spec")[metrics].mean()

,KLD,JSD,L1_norm,L2_norm,LINF_norm
etahat_fc_model_spec,,,,,
VAR,2.272059,0.165739,91423.948742,6240.228457,971.249821
adalasso,2.353588,0.170925,93418.536205,6347.826299,985.601194
ridge,2.353644,0.170930,93420.070799,6347.911892,985.611830


In [74]:
df_loss_base_long.groupby("forecast_dFPC_dimension")[metrics].mean().sort_values(by="KLD")

,KLD,JSD,L1_norm,L2_norm,LINF_norm
forecast_dFPC_dimension,,,,,
2,2.32643,0.169198,92754.185248,6311.988883,980.820948


In [75]:
df_loss_base_long.groupby("etahat_fc_model_lags")[metrics].mean().sort_values(by="KLD")

,KLD,JSD,L1_norm,L2_norm,LINF_norm
etahat_fc_model_lags,,,,,
2,2.320603,0.168823,92626.654535,6305.423543,980.032281
1,2.332257,0.169573,92881.715962,6318.554222,981.609615


In [76]:
df_loss_base_long.groupby(["etahat_fc_model_spec", "etahat_fc_model_lags"])[metrics].mean().sort_values(by="KLD")

KLD       JSD       L1_norm  \
etahat_fc_model_spec etahat_fc_model_lags                                     
VAR                  2                     2.256540  0.164690  91077.308992   
                     1                     2.287579  0.166788  91770.588491   
adalasso             2                     2.352593  0.170886  93400.203718   
ridge                2                     2.352677  0.170892  93402.450896   
adalasso             1                     2.354582  0.170964  93436.868691   
ridge                1                     2.354611  0.170967  93437.690703   

                                               L2_norm   LINF_norm  
etahat_fc_model_spec etahat_fc_model_lags                           
VAR                  2                     6222.049285  969.090949  
                     1                     6258.407630  973.408693  
adalasso             2                     6347.049030  985.495224  
ridge                2                     6347.172315  985.510670  
adalasso             1                     6348.603567  985.707163  
ridge                1                     6348.651469  985.712990